# S03 — Sémantique : RDF, OWL et SPARQL sur NYC Taxi

Jusqu'ici, nos données sont des tables. Une table dit *ce que contient* une colonne, mais pas *ce que signifie* une ligne :
rien n'indique que « Manhattan » est un arrondissement, ni qu'une zone appartient à un arrondissement.

Aujourd'hui, nous allons écrire ce sens explicitement, sous une forme que n'importe quel système peut lire :

1. décrire notre domaine avec une **ontologie OWL** (les classes et leurs relations) ;
2. y charger nos données sous forme de **triplets RDF** ;
3. les interroger avec **SPARQL** ;
4. les relier à une base de connaissances publique (**Wikidata**) ;
5. visualiser le **graphe de connaissances** obtenu.

In [ ]:
pip install -q rdflib pyvis

In [ ]:
import sys; sys.path.append("/home/jovyan/work")
from commun.spark_utils import RAW

import pandas as pd
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD

DOSSIER = "/home/jovyan/work/seances/S03-semantique"
NYC = Namespace("http://example.org/nyctaxi#")   # l'espace de noms de notre projet
WD = Namespace("http://www.wikidata.org/entity/")  # Wikidata

g = Graph()
g.bind("nyc", NYC); g.bind("owl", OWL); g.bind("wd", WD)

## 1. Un triplet, c'est une phrase

Tout, en RDF, s'écrit sous la forme **sujet → prédicat → objet** :

| Sujet | Prédicat | Objet |
|---|---|---|
| zone 132 | est un | Zone |
| zone 132 | s'appelle | « JFK Airport » |
| zone 132 | est située dans | Queens |

Chaque élément est identifié par une adresse unique (une URI). C'est ce qui permet à deux systèmes qui ne se connaissent pas
de parler de la même chose.

## 2. L'ontologie : le vocabulaire du domaine

On déclare d'abord les **classes** (les types de choses) et les **propriétés** (les relations entre elles).

In [ ]:
def classe(nom, libelle):
    g.add((NYC[nom], RDF.type, OWL.Class))
    g.add((NYC[nom], RDFS.label, Literal(libelle, lang="fr")))

def relation(nom, domaine, cible, libelle):
    g.add((NYC[nom], RDF.type, OWL.ObjectProperty))
    g.add((NYC[nom], RDFS.domain, NYC[domaine]))
    g.add((NYC[nom], RDFS.range, NYC[cible]))
    g.add((NYC[nom], RDFS.label, Literal(libelle, lang="fr")))

def attribut(nom, domaine, type_xsd, libelle):
    g.add((NYC[nom], RDF.type, OWL.DatatypeProperty))
    g.add((NYC[nom], RDFS.domain, NYC[domaine]))
    g.add((NYC[nom], RDFS.range, type_xsd))
    g.add((NYC[nom], RDFS.label, Literal(libelle, lang="fr")))

# Classes
classe("Arrondissement", "arrondissement")
classe("Zone", "zone de taxi")
classe("Course", "course de taxi")
classe("ModeDePaiement", "mode de paiement")
classe("ZoneAeroport", "zone d'aéroport")
g.add((NYC.ZoneAeroport, RDFS.subClassOf, NYC.Zone))   # un aéroport EST une zone

# Relations entre objets
relation("situeeDans", "Zone", "Arrondissement", "est située dans")
relation("departDe", "Course", "Zone", "part de")
relation("arriveA", "Course", "Zone", "arrive à")
relation("payeePar", "Course", "ModeDePaiement", "est payée par")

# Attributs (valeurs)
attribut("idZone", "Zone", XSD.integer, "identifiant TLC de la zone")
attribut("nom", "Zone", XSD.string, "nom")
attribut("montantTotal", "Course", XSD.decimal, "montant total ($)")
attribut("distanceMiles", "Course", XSD.decimal, "distance (miles)")

print(len(g), "triplets dans l'ontologie")

In [ ]:
# Le format Turtle : la façon lisible d'écrire des triplets
print(g.serialize(format="turtle")[:1500])

## 3. Les données : arrondissements, zones, modes de paiement

On transforme maintenant le référentiel des zones (un CSV) en triplets. Les codes de paiement
viennent du dictionnaire de données de la TLC.

In [ ]:
zones = pd.read_csv(f"{RAW}/taxi_zone_lookup.csv")
zones = zones[~zones["Borough"].isin(["Unknown", "N/A"]) & zones["Borough"].notna()]

def uri_arr(nom):
    return NYC["arr_" + nom.replace(" ", "_")]

for arr in zones["Borough"].unique():
    g.add((uri_arr(arr), RDF.type, NYC.Arrondissement))
    g.add((uri_arr(arr), RDFS.label, Literal(arr)))

for _, z in zones.iterrows():
    u = NYC[f"zone_{z.LocationID}"]
    est_aeroport = "Airport" in str(z.Zone)
    g.add((u, RDF.type, NYC.ZoneAeroport if est_aeroport else NYC.Zone))
    g.add((u, NYC.idZone, Literal(int(z.LocationID), datatype=XSD.integer)))
    g.add((u, NYC.nom, Literal(z.Zone)))
    g.add((u, NYC.situeeDans, uri_arr(z.Borough)))

paiements = {1: "Carte bancaire", 2: "Espèces", 3: "Gratuit", 4: "Litige", 5: "Inconnu", 6: "Course annulée"}
for code, libelle in paiements.items():
    g.add((NYC[f"paiement_{code}"], RDF.type, NYC.ModeDePaiement))
    g.add((NYC[f"paiement_{code}"], RDFS.label, Literal(libelle, lang="fr")))

print(len(g), "triplets au total ;", len(zones), "zones chargées")

Pour garder un graphe de taille raisonnable, on n'y met qu'un **échantillon de 300 courses**.
Les millions de courses restent dans le data lake : le graphe sert à décrire et relier, pas à stocker le volume.

In [ ]:
MOIS = "2025-01"
cols = ["PULocationID", "DOLocationID", "payment_type", "total_amount", "trip_distance"]
courses = (pd.read_parquet(f"{RAW}/yellow_tripdata_{MOIS}.parquet", columns=cols)
             .query("total_amount > 0 and trip_distance > 0")
             .sample(300, random_state=42)
             .reset_index(drop=True))

ids_valides = set(zones["LocationID"])
for i, c in courses.iterrows():
    if c.PULocationID not in ids_valides or c.DOLocationID not in ids_valides:
        continue
    u = NYC[f"course_{i}"]
    g.add((u, RDF.type, NYC.Course))
    g.add((u, NYC.departDe, NYC[f"zone_{int(c.PULocationID)}"]))
    g.add((u, NYC.arriveA, NYC[f"zone_{int(c.DOLocationID)}"]))
    if int(c.payment_type) in paiements:
        g.add((u, NYC.payeePar, NYC[f"paiement_{int(c.payment_type)}"]))
    g.add((u, NYC.montantTotal, Literal(round(float(c.total_amount), 2), datatype=XSD.decimal)))
    g.add((u, NYC.distanceMiles, Literal(round(float(c.trip_distance), 2), datatype=XSD.decimal)))

print(len(g), "triplets au total")

## 4. Interroger le graphe avec SPARQL

SPARQL ressemble à SQL, mais au lieu de tables on décrit un **motif de triplets** à retrouver dans le graphe.
Les mots qui commencent par `?` sont des variables.

In [ ]:
PREFIXES = """
PREFIX nyc:  <http://example.org/nyctaxi#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

def sparql(requete):
    res = g.query(PREFIXES + requete)
    return pd.DataFrame([[str(v) if v is not None else None for v in ligne] for ligne in res],
                        columns=[str(v) for v in res.vars])

# Requête 1 : les zones de Manhattan
sparql("""
SELECT ?nom WHERE {
  ?zone nyc:situeeDans ?arr ;
        nyc:nom ?nom .
  ?arr rdfs:label "Manhattan" .
}
ORDER BY ?nom
""").head(10)

In [ ]:
# Requête 2 : nombre de zones par arrondissement
sparql("""
SELECT ?arrondissement (COUNT(?zone) AS ?nb_zones) WHERE {
  ?zone nyc:situeeDans ?arr .
  ?arr rdfs:label ?arrondissement .
}
GROUP BY ?arrondissement
ORDER BY DESC(?nb_zones)
""")

La requête suivante montre l'intérêt de l'ontologie. On demande les courses parties d'une `ZoneAeroport`,
alors que les données ne disent jamais « aéroport » sur les courses : le graphe fait le lien tout seul.

In [ ]:
# Requête 3 : courses parties d'un aéroport, montant moyen
sparql("""
SELECT ?aeroport (COUNT(?c) AS ?courses) (AVG(?m) AS ?montant_moyen) WHERE {
  ?c a nyc:Course ;
     nyc:departDe ?z ;
     nyc:montantTotal ?m .
  ?z a nyc:ZoneAeroport ;
     nyc:nom ?aeroport .
}
GROUP BY ?aeroport
""").astype({"courses": int, "montant_moyen": float}).round(2)

## 5. Interopérabilité : se relier au reste du web

Notre « Manhattan » n'a de sens que dans notre projet. En le déclarant identique (`owl:sameAs`)
à l'entité Manhattan de Wikidata, n'importe quel autre système qui utilise Wikidata peut croiser ses données avec les nôtres
(population, superficie, etc.) sans aucune table de correspondance.

In [ ]:
wikidata = {"Manhattan": "Q11299", "Brooklyn": "Q18419", "Queens": "Q18424",
            "Bronx": "Q18426", "Staten Island": "Q18432"}
for arr, qid in wikidata.items():
    if (uri_arr(arr), RDF.type, NYC.Arrondissement) in g:
        g.add((uri_arr(arr), OWL.sameAs, WD[qid]))

sparql("""
PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?arrondissement ?wikidata WHERE {
  ?arr owl:sameAs ?wikidata ; rdfs:label ?arrondissement .
}
""")

## 6. Enregistrer l'ontologie et le graphe

In [ ]:
# Ontologie seule (le vocabulaire) et graphe complet (vocabulaire + données)
onto = Graph(); onto.bind("nyc", NYC); onto.bind("owl", OWL)
for s, p, o in g:
    if (s, RDF.type, OWL.Class) in g or (s, RDF.type, OWL.ObjectProperty) in g or (s, RDF.type, OWL.DatatypeProperty) in g:
        onto.add((s, p, o))

onto.serialize(f"{DOSSIER}/ontologie_nyctaxi.ttl", format="turtle")
g.serialize(f"{DOSSIER}/graphe_nyctaxi.ttl", format="turtle")
print("ontologie_nyctaxi.ttl :", len(onto), "triplets")
print("graphe_nyctaxi.ttl    :", len(g), "triplets")

## 7. Visualiser le graphe de connaissances

On dessine les arrondissements et leurs zones (les aéroports sont représentés par une étoile).
Survolez un point pour lire le nom de la zone ; les boutons en bas de la page permettent de zoomer et de recentrer.
Le résultat est une page web interactive : on peut zoomer, déplacer les nœuds et survoler les noms.

In [ ]:
from pyvis.network import Network

couleurs = {"Manhattan": "#4C72B0", "Brooklyn": "#DD8452", "Queens": "#55A868",
            "Bronx": "#C44E52", "Staten Island": "#8172B3", "EWR": "#937860"}

net = Network(height="800px", width="100%", bgcolor="#ffffff", cdn_resources="in_line")
for arr in zones["Borough"].unique():
    nb = int((zones["Borough"] == arr).sum())
    net.add_node(f"arr_{arr}", label=arr, title=f"{arr} : {nb} zones", shape="dot", size=45,
                 color=couleurs.get(arr, "#999999"),
                 font={"size": 46, "face": "arial", "color": "#222222", "strokeWidth": 6, "strokeColor": "#ffffff"})

for _, z in zones.iterrows():
    aeroport = "Airport" in str(z.Zone)
    net.add_node(int(z.LocationID),
                 label=z.Zone if aeroport else "",
                 title=f"{z.Zone} ({z.Borough})",
                 shape="star" if aeroport else "dot",
                 size=28 if aeroport else 9,
                 color="#222222" if aeroport else couleurs.get(z.Borough, "#999999"),
                 font={"size": 30, "face": "arial", "color": "#222222", "strokeWidth": 5, "strokeColor": "#ffffff"})
    net.add_edge(int(z.LocationID), f"arr_{z.Borough}", color="#cccccc", width=0.6)

net.set_options('''{
  "physics": {"solver": "forceAtlas2Based",
              "forceAtlas2Based": {"gravitationalConstant": -45, "centralGravity": 0.02,
                                   "springLength": 35, "springConstant": 0.09, "avoidOverlap": 0.4},
              "stabilization": {"iterations": 500}},
  "interaction": {"hover": true, "navigationButtons": true, "tooltipDelay": 100}
}''')
chemin = f"{DOSSIER}/graphe_zones.html"
net.write_html(chemin)
print("Graphe enregistré :", chemin)

In [ ]:
# Affichage dans le notebook (JupyterLab). Dans VS Code, ouvrez plutôt graphe_zones.html dans le navigateur.
from IPython.display import HTML
HTML(open(chemin, encoding="utf-8").read())

## À retenir

- Une **ontologie** décrit le vocabulaire d'un domaine : classes, relations, hiérarchies (`ZoneAeroport` ⊂ `Zone`).
- Les données deviennent des **triplets RDF**, interrogeables avec **SPARQL**.
- `owl:sameAs` relie nos données à des référentiels publics : c'est le cœur de l'**interopérabilité**.
- Le graphe ne remplace pas le data lake : il **décrit et relie** les données, le data lake **stocke** le volume.

## Exercices

1. Ajoutez une classe `ZoneDeService` (valeurs « Yellow Zone », « Boro Zone », « Airports », colonne `service_zone` du CSV)
   et une relation `aPourServiceZone`. Combien de zones sont en « Boro Zone » ?
2. Écrivez une requête SPARQL qui donne, pour chaque mode de paiement, le nombre de courses de l'échantillon et leur montant moyen.
3. Écrivez une requête qui liste les courses dont le départ et l'arrivée sont dans **deux arrondissements différents**.